# 30. Visualization Test

- Goal: build a PPT-friendly recording-view pose playback from normalized coordinates.
- Docs: `docs_eng/pipeline/10_visualization.md` / `docs/pipeline/10_visualization.md`
- Inputs: p01 squat pose/annotation files when available, otherwise saved processed pose output.
- Outputs: Interactive Plotly figure and optional HTML under `data/processed/visualization/`.
- Checks: First 10 reps can be played/paused in recording-view using `norm` coordinates.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing pyproject.toml")
    PROJECT_ROOT = PROJECT_ROOT.parent

from movement.config import LANDMARKS, CONNECTIONS
from movement.io import load_pose_csv
from movement.pipeline import NormalizationConfig, PreprocessingConfig, run_pipeline
from movement.stage_context import build_stage_check_pipeline_config
from movement.visualization import create_pose_animation

RECORDING_ID = "p01_squat_set1"
POSE_CSV = PROJECT_ROOT / "data/pose/mediapipe/no_consent/20260517/p01_squat_set1_output_pose.csv"
ANNOTATION_CSV = POSE_CSV.with_name("p01_squat_set1_annotation.csv")
PROCESSED_POSE_CSV = PROJECT_ROOT / "data/processed/user_movement_evaluation/processed_pose.csv"
OUTPUT_DIR = PROJECT_ROOT / "data/processed/visualization"
OUTPUT_HTML = OUTPUT_DIR / f"{RECORDING_ID}_norm_recording_view_10rep.html"

TARGET_REP_COUNT = 10
COORD_MODE = "norm"
PLOT_WIDTH = 1280
PLOT_HEIGHT = 720

if POSE_CSV.exists() and ANNOTATION_CSV.exists():
    raw_df = load_pose_csv(POSE_CSV)
    cfg = build_stage_check_pipeline_config(
        exercise_id="squat",
        definitions_dir=PROJECT_ROOT / "data/definitions/exercises",
        annotation_csv=ANNOTATION_CSV,
        preprocessing_config=PreprocessingConfig(enabled=True),
        normalization_config=NormalizationConfig(
            enabled=True,
            keep_reference_columns=True,
        ),
        enable_validation=True,
        enable_annotation=True,
        enable_preprocessing=True,
        enable_normalization=True,
        enable_canonicalization=False,
        enable_rep_segmentation=True,
        enable_phase_segmentation=True,
        enable_features=False,
        enable_biomech=False,
        enable_biomarker=False,
    )
    df, report = run_pipeline(raw_df, cfg)
    data_source = POSE_CSV.relative_to(PROJECT_ROOT)
elif PROCESSED_POSE_CSV.exists():
    df = pd.read_csv(PROCESSED_POSE_CSV)
    report = {}
    data_source = PROCESSED_POSE_CSV.relative_to(PROJECT_ROOT)
else:
    raise FileNotFoundError(
        "Missing p01 pose/annotation inputs and processed pose fallback. "
        "Run the user-evaluation or stage-check pipeline first."
    )

norm_cols = [f"{landmark}_{COORD_MODE}_x" for landmark in LANDMARKS]
missing_norm_cols = [col for col in norm_cols if col not in df.columns]
if missing_norm_cols:
    raise ValueError(f"Missing normalized coordinate columns: {missing_norm_cols[:6]}")

rep_id_int = pd.to_numeric(df["rep_id"], errors="coerce")
rep_mask = df["segment_type"].eq("rep") & rep_id_int.notna()
rep_ids = sorted(rep_id_int.loc[rep_mask].astype(int).unique().tolist())
if len(rep_ids) < TARGET_REP_COUNT:
    raise ValueError(
        f"Need at least {TARGET_REP_COUNT} reps for this PPT view; found {len(rep_ids)}."
    )

target_rep_ids = rep_ids[:TARGET_REP_COUNT]
target_rep_rows = df.loc[rep_mask & rep_id_int.isin(target_rep_ids)].copy()
start_frame = int(target_rep_rows["frame"].min())
end_frame = int(target_rep_rows["frame"].max())
plot_df = df.loc[df["frame"].between(start_frame, end_frame)].copy()
plot_df = plot_df.sort_values("frame").reset_index(drop=True)

def estimate_recorded_duration_s(dataframe):
    if "timestamp" not in dataframe.columns or len(dataframe) < 2:
        return None
    timestamps = pd.to_numeric(dataframe["timestamp"], errors="coerce").dropna()
    if len(timestamps) < 2:
        return None
    duration_s = float(timestamps.iloc[-1] - timestamps.iloc[0])
    return duration_s if duration_s > 0 else None

recorded_duration_s = estimate_recorded_duration_s(plot_df)

summary = pd.DataFrame(
    [
        {"item": "data_source", "value": str(data_source)},
        {"item": "coord_mode", "value": COORD_MODE},
        {"item": "target_reps", "value": f"{target_rep_ids[0]}-{target_rep_ids[-1]}"},
        {"item": "frame_span", "value": f"{start_frame}-{end_frame}"},
        {"item": "frames", "value": len(plot_df)},
        {
            "item": "recorded_duration_s",
            "value": None if recorded_duration_s is None else round(recorded_duration_s, 3),
        },
        {"item": "html_export", "value": str(OUTPUT_HTML.relative_to(PROJECT_ROOT))},
    ]
)
display(summary)

def estimate_frame_duration_ms(dataframe, recorded_duration_s, default_ms=33.0):
    if recorded_duration_s is not None and len(dataframe) > 1:
        return max(1.0, (recorded_duration_s / (len(dataframe) - 1)) * 1000)
    if "timestamp" not in dataframe.columns:
        return default_ms
    dt = pd.to_numeric(dataframe["timestamp"], errors="coerce").diff().dropna()
    if dt.empty:
        return default_ms
    median_dt = float(dt.median())
    if median_dt <= 0:
        return default_ms
    return max(1.0, median_dt * 1000)

frame_duration_ms = estimate_frame_duration_ms(plot_df, recorded_duration_s)
estimated_playback_s = (len(plot_df) - 1) * frame_duration_ms / 1000
print(
    f"playback frame duration: {frame_duration_ms:.3f} ms "
    f"(~{1000 / frame_duration_ms:.1f} fps, estimated {estimated_playback_s:.3f} s)"
)

fig = create_pose_animation(
    df=plot_df,
    landmarks=LANDMARKS,
    connections=CONNECTIONS,
    coord_mode=COORD_MODE,
    title=f"{RECORDING_ID} norm pose - recording-view 10 reps",
    show_text=False,
    frame_duration=frame_duration_ms,
    width=PLOT_WIDTH,
    height=PLOT_HEIGHT,
)

recording_view_camera = dict(
    eye=dict(x=0.0, y=-2.5, z=0.0),
    center=dict(x=0.0, y=0.0, z=0.0),
    up=dict(x=0.0, y=0.0, z=1.0),
    projection=dict(type="orthographic"),
)
fig.update_layout(scene_camera=recording_view_camera)
fig.update_layout(
    margin=dict(l=0, r=0, t=48, b=0),
    paper_bgcolor="white",
    plot_bgcolor="white",
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig.write_html(OUTPUT_HTML, include_plotlyjs=True, full_html=True)
print("saved html:", OUTPUT_HTML.relative_to(PROJECT_ROOT))

fig.show()
